# RFM 세그먼트 분석 과제
## 과제 목표:
- 아래 표는 RFM으로 고객군을 나누는 예시를 보여줍니다.
- 본인이 생각하는 세그먼트 전략에 따라 나눠주세요. (아래 분류대로 되지 않아도되며 최소 4개의 세그먼트를 나눠주세요.)
- 각 고객군 rfm_df 데이터를 탐색하여 냐눠서 rfm_df['segmentation] 열에 저장해주세요.
- 본인의 세그먼트 전략의 근거를 간략히 서술해주세요. (200자 내외)
- 제출자료 : rfm_df 를 csv파일로 제출. 주피터노트북은 html로 변환하여 함께 제출.

In [17]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

## 1. 데이터 로드 및 전처리

In [18]:
# 데이터 로드
path = '../class/datasets/ml/crm/marketing_campaign.csv'
df = pd.read_csv(path, sep='\t')
print(f"데이터 크기: {df.shape}")
df.head()

데이터 크기: (2240, 29)


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,...,5,0,0,0,0,0,0,3,11,0


In [19]:
# 데이터 정보 확인
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   2240 non-null   int64  
 1   Year_Birth           2240 non-null   int64  
 2   Education            2240 non-null   object 
 3   Marital_Status       2240 non-null   object 
 4   Income               2216 non-null   float64
 5   Kidhome              2240 non-null   int64  
 6   Teenhome             2240 non-null   int64  
 7   Dt_Customer          2240 non-null   object 
 8   Recency              2240 non-null   int64  
 9   MntWines             2240 non-null   int64  
 10  MntFruits            2240 non-null   int64  
 11  MntMeatProducts      2240 non-null   int64  
 12  MntFishProducts      2240 non-null   int64  
 13  MntSweetProducts     2240 non-null   int64  
 14  MntGoldProds         2240 non-null   int64  
 15  NumDealsPurchases    2240 non-null   i

In [20]:
# Income 컬럼의 결측치를 중앙값으로 대체
df['Income'].fillna(df['Income'].median(), inplace=True)
print(f"결측치 개수: {df.isnull().sum().sum()}")

결측치 개수: 0


## 2. RFM 지표 생성

In [21]:
# rfm_df 생성
rfm_df = pd.DataFrame()
rfm_df['ID'] = df['ID']

# Recency 값은 기존 컬럼을 그대로 사용
rfm_df['Recency'] = df['Recency']

# Frequency 계산 (모든 구매 채널의 횟수 합산)
freq_cols = ['NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases']
rfm_df['Frequency'] = df[freq_cols].sum(axis=1)

# Monetary 계산 (모든 제품 카테고리 구매액 합산)
monetary_cols = ['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
rfm_df['Monetary'] = df[monetary_cols].sum(axis=1)

print("생성된 RFM 데이터프레임:")
rfm_df.head()

생성된 RFM 데이터프레임:


,ID,Recency,Frequency,Monetary
0,5524,58,22,1617
1,2174,38,4,27
2,4141,26,20,776
3,6182,26,6,53
4,5324,94,14,422


In [22]:
import plotly.offline as pyo
pyo.init_notebook_mode(connected=True)

## 3. RFM 데이터 탐색 및 분포 확인

In [23]:
# 통계량 확인
print("RFM 지표 기본 통계량:")
rfm_df[['Recency', 'Frequency', 'Monetary']].describe()

RFM 지표 기본 통계량:


,Recency,Frequency,Monetary
count,2240.000000,2240.000000,2240.000000
mean,49.109375,12.537054,605.798214
std,28.962453,7.205741,602.249288
min,0.000000,0.000000,5.000000
25%,24.000000,6.000000,68.750000
50%,49.000000,12.000000,396.000000
75%,74.000000,18.000000,1045.500000
max,99.000000,32.000000,2525.000000


In [24]:
# RFM 지표별 분포 시각화
fig = make_subplots(rows=1, cols=3, 
                    subplot_titles=('Recency 분포', 'Frequency 분포', 'Monetary 분포'))

fig.add_trace(go.Histogram(x=rfm_df['Recency'], name='Recency', nbinsx=20), row=1, col=1)
fig.add_trace(go.Histogram(x=rfm_df['Frequency'], name='Frequency', nbinsx=20), row=1, col=2)
fig.add_trace(go.Histogram(x=rfm_df['Monetary'], name='Monetary', nbinsx=20), row=1, col=3)

fig.update_layout(title_text="RFM 지표별 분포", height=400, showlegend=False)
fig.show()

## 4. RFM 스코어 부여

In [25]:
# R, F, M 값을 기준으로 5분위수로 나누어 점수 부여 (1-5점)
r_labels = range(5, 0, -1)  # Recency는 낮을수록 좋음 (5,4,3,2,1)
f_labels = range(1, 6)      # Frequency는 높을수록 좋음 (1,2,3,4,5)
m_labels = range(1, 6)      # Monetary는 높을수록 좋음 (1,2,3,4,5)

# qcut을 사용하여 분위수 기반 스코어링
rfm_df['R_Score'] = pd.qcut(rfm_df['Recency'], q=5, labels=r_labels, duplicates='drop')
rfm_df['F_Score'] = pd.qcut(rfm_df['Frequency'], q=5, labels=f_labels, duplicates='drop')
rfm_df['M_Score'] = pd.qcut(rfm_df['Monetary'], q=5, labels=m_labels, duplicates='drop')

# 전체 RFM 점수 계산
rfm_df['RFM_Score'] = rfm_df['R_Score'].astype(int) + rfm_df['F_Score'].astype(int) + rfm_df['M_Score'].astype(int)

print("RFM 스코어 부여 완료:")
rfm_df[['ID', 'Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'RFM_Score']].head()

RFM 스코어 부여 완료:


,ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
0,5524,58,22,1617,3,5,5,13
1,2174,38,4,27,4,1,1,6
2,4141,26,20,776,4,4,4,12
3,6182,26,6,53,4,2,1,7
4,5324,94,14,422,1,3,3,7


In [26]:
# RFM 총점 분포 확인
print("RFM 총점 분포:")
print(rfm_df['RFM_Score'].value_counts().sort_index())

fig = px.histogram(rfm_df, x='RFM_Score', title='RFM 총점 분포', 
                   labels={'RFM_Score': 'RFM 총점', 'count': '고객수'})
fig.show()

RFM 총점 분포:
RFM_Score
3      81
4     106
5     184
6     192
7     242
8     194
9     236
10    217
11    280
12    189
13    163
14    131
15     25
Name: count, dtype: int64


## 5. 고객 세그먼트 정의 및 분류

In [27]:
# 세그먼트 전략 정의
def assign_segmentation(row):
    r_score = int(row['R_Score'])
    f_score = int(row['F_Score'])
    m_score = int(row['M_Score'])
    total_score = row['RFM_Score']
    
    # VIP 고객: 모든 지표가 우수 (R>=4, F>=4, M>=4)
    if r_score >= 4 and f_score >= 4 and m_score >= 4:
        return 'VIP 고객'
    
    # 충성 고객: 구매빈도와 구매금액이 높고 최신성도 양호 (F>=4, M>=4, R>=3)
    elif f_score >= 4 and m_score >= 4 and r_score >= 3:
        return '충성 고객'
    
    # 신규 고객: 최신성은 높지만 구매빈도나 구매금액이 낮음 (R>=4, F<=2 or M<=2)
    elif r_score >= 4 and (f_score <= 2 or m_score <= 2):
        return '신규 고객'
    
    # 잠재 고객: 중간 수준의 지표들 (총점 9-11점)
    elif 9 <= total_score <= 11:
        return '잠재 고객'
    
    # 관심 필요: 구매빈도나 구매금액은 있지만 최신성이 낮음 (R<=2, F>=3 or M>=3)
    elif r_score <= 2 and (f_score >= 3 or m_score >= 3):
        return '관심 필요'
    
    # 이탈 위험: 모든 지표가 낮음 (총점 7점 이하)
    elif total_score <= 7:
        return '이탈 위험'
    
    # 일반 고객: 나머지
    else:
        return '일반 고객'

# 세그먼트 분류 적용
rfm_df['segmentation'] = rfm_df.apply(assign_segmentation, axis=1)

print("세그먼트 분류 완료:")
print(rfm_df['segmentation'].value_counts())

세그먼트 분류 완료:
segmentation
잠재 고객     610
이탈 위험     528
신규 고객     380
VIP 고객    292
관심 필요     203
충성 고객     159
일반 고객      68
Name: count, dtype: int64


## 6. 세그먼트별 특성 분석

In [28]:
# 세그먼트별 RFM 지표 평균 분석
segment_analysis = rfm_df.groupby('segmentation').agg({
    'Recency': ['mean', 'std'],
    'Frequency': ['mean', 'std'],
    'Monetary': ['mean', 'std'],
    'RFM_Score': 'mean',
    'ID': 'count'
}).round(2)

segment_analysis.columns = ['최신성_평균', '최신성_표준편차', '구매빈도_평균', '구매빈도_표준편차',
                           '구매금액_평균', '구매금액_표준편차', 'RFM점수_평균', '고객수']

print("세그먼트별 특성 분석:")
segment_analysis

세그먼트별 특성 분석:


,최신성_평균,최신성_표준편차,구매빈도_평균,구매빈도_표준편차,구매금액_평균,구매금액_표준편차,RFM점수_평균,고객수
segmentation,,,,,,,,
VIP 고객,18.97,11.07,20.99,3.29,1234.92,429.31,13.50,292
관심 필요,79.12,11.75,14.14,4.88,607.54,483.45,8.19,203
신규 고객,19.43,11.62,5.17,1.67,73.24,62.80,7.48,380
이탈 위험,69.37,17.44,5.16,1.57,68.58,45.20,4.92,528
일반 고객,14.29,14.19,14.74,3.24,1008.66,545.96,11.91,68
잠재 고객,58.11,27.26,16.54,4.41,888.52,518.26,10.14,610
충성 고객,50.16,5.90,20.79,3.29,1247.96,390.08,12.03,159


In [29]:
# 세그먼트별 분포 시각화
segment_counts = rfm_df['segmentation'].value_counts().reset_index()
segment_counts.columns = ['Segment', 'Count']

fig_bar = px.bar(segment_counts, x='Segment', y='Count',
                 title='고객 세그먼트별 분포',
                 color='Segment',
                 text_auto=True)
fig_bar.update_xaxes(categoryorder='total descending')
fig_bar.show()

In [30]:
# 3D 산점도 시각화
fig_3d = px.scatter_3d(rfm_df, x='Recency', y='Frequency', z='Monetary',
                       color='segmentation', size='RFM_Score',
                       title='RFM 세그먼트별 3차원 분포',
                       labels={'Recency': '최신성 (일)',
                              'Frequency': '구매빈도 (회)',
                              'Monetary': '구매금액 (원)'})
fig_3d.show()

In [31]:
# 세그먼트별 매출 기여도 분석
segment_value = rfm_df.groupby('segmentation').agg({
    'Monetary': ['sum', 'mean', 'count']
}).round(2)

segment_value.columns = ['총_구매금액', '평균_구매금액', '고객수']
segment_value['매출_기여율'] = (segment_value['총_구매금액'] / segment_value['총_구매금액'].sum() * 100).round(1)
segment_value['고객_비율'] = (segment_value['고객수'] / segment_value['고객수'].sum() * 100).round(1)

print("세그먼트별 비즈니스 기여도:")
segment_value

세그먼트별 비즈니스 기여도:


,총_구매금액,평균_구매금액,고객수,매출_기여율,고객_비율
segmentation,,,,,
VIP 고객,360598,1234.92,292,26.6,13.0
관심 필요,123331,607.54,203,9.1,9.1
신규 고객,27833,73.24,380,2.1,17.0
이탈 위험,36212,68.58,528,2.7,23.6
일반 고객,68589,1008.66,68,5.1,3.0
잠재 고객,542000,888.52,610,39.9,27.2
충성 고객,198425,1247.96,159,14.6,7.1


## 7. 세그먼트 전략 근거 및 분석 결과

In [32]:
# 최종 결과 확인
print("최종 RFM 세그먼트 분석 결과:")
print(f"총 고객수: {len(rfm_df)}명")
print(f"세그먼트 수: {rfm_df['segmentation'].nunique()}개")
print("\n세그먼트별 분포:")
for segment in rfm_df['segmentation'].value_counts().index:
    count = rfm_df['segmentation'].value_counts()[segment]
    ratio = count / len(rfm_df) * 100
    print(f"- {segment}: {count}명 ({ratio:.1f}%)")

최종 RFM 세그먼트 분석 결과:
총 고객수: 2240명
세그먼트 수: 7개

세그먼트별 분포:
- 잠재 고객: 610명 (27.2%)
- 이탈 위험: 528명 (23.6%)
- 신규 고객: 380명 (17.0%)
- VIP 고객: 292명 (13.0%)
- 관심 필요: 203명 (9.1%)
- 충성 고객: 159명 (7.1%)
- 일반 고객: 68명 (3.0%)


## 8. 데이터 저장

In [33]:
# CSV 파일로 저장
rfm_df.to_csv('rfm_segmentation_result.csv', index=False, encoding='utf-8-sig')
print("RFM 분석 결과가 'rfm_segmentation_result.csv' 파일로 저장되었습니다.")

# 최종 데이터프레임 확인
print("\n저장된 데이터 컬럼:")
print(rfm_df.columns.tolist())
print("\n데이터 샘플:")
rfm_df[['ID', 'Recency', 'Frequency', 'Monetary', 'RFM_Score', 'segmentation']].head(10)

RFM 분석 결과가 'rfm_segmentation_result.csv' 파일로 저장되었습니다.

저장된 데이터 컬럼:
['ID', 'Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'RFM_Score', 'segmentation']

데이터 샘플:


,ID,Recency,Frequency,Monetary,RFM_Score,segmentation
0,5524,58,22,1617,13,충성 고객
1,2174,38,4,27,6,신규 고객
2,4141,26,20,776,12,VIP 고객
3,6182,26,6,53,7,신규 고객
4,5324,94,14,422,7,관심 필요
5,7446,16,20,716,13,VIP 고객
6,965,34,17,590,11,잠재 고객
7,6177,32,8,169,8,신규 고객
8,4855,19,5,46,7,신규 고객
9,5899,68,1,49,4,이탈 위험


## RFM 분석 결과 종합 보고서

### 세그먼트 전략 근거
>본 분석에서는 RFM 점수와 개별 지표를 종합적으로 고려하여 7개 세그먼트로 분류  
>VIP 고객은 모든 지표가 우수한 최상위층, 충성 고객은 구매력과 충성도가 높은 핵심층, 신규 고객은 최근 유입된 성장 가능층으로 정의  
>잠재 고객은 중간 수준의 균형잡힌 그룹, 관심 필요는 과거 구매력은 있지만 재활성화가 필요한 그룹, 이탈 위험은 즉시 개입이 필요한 그룹으로 구분  
>이러한 분류를 통해 각 세그먼트의 특성에 맞는 차별화된 마케팅 전략 수립이 가능하다고 판단  

### 1. 고객 세그먼트 분포 현황

- **잠재 고객 (27.2%, 610명)**: 전체 고객의 가장 큰 비중을 차지하는 그룹으로, 중간 수준의 RFM 지표를 보이며 향후 성장 가능성이 높은 핵심 타겟층
- **이탈 위험 (23.6%, 528명)**: 두 번째로 큰 그룹으로, 모든 RFM 지표가 낮아 즉각적인 관리가 필요한 상황
- **신규 고객 (17.0%, 380명)**: 최신성은 높지만 구매빈도나 구매금액이 낮은 최근 유입된 고객층
- **VIP 고객 (13.0%, 292명)**: 모든 RFM 지표가 우수한 최상위 고객층으로, 비중은 작지만 높은 가치를 지님
- **관심 필요 (9.1%, 203명)**: 과거 구매력은 있지만 최신성이 낮아 재활성화가 필요한 그룹
- **충성 고객 (7.1%, 159명)**: 구매빈도와 구매금액이 높고 최신성도 양호한 안정적인 수익 기반층
- **일반 고객 (3.0%, 68명)**: 가장 작은 비중을 차지하는 중간 수준의 고객층

### 2. RFM 지표별 특성 분석

**2.1 Recency (최신성) 지표**
- VIP 고객과 신규 고객이 가장 최근 구매 활동을 보이며 높은 참여도를 나타내고, 이탈 위험과 관심 필요 그룹은 상대적으로 오래된 구매 이력을 보여 재활성화 전략이 필요

**2.2 Frequency (구매빈도) 지표**
- VIP 고객과 충성 고객이 높은 구매 빈도를 보이며 꾸준한 거래 관계를 유지하고 있으며, 신규 고객과 이탈 위험 그룹은 낮은 구매 빈도를 보여 참여 증대 방안이 필요

**2.3 Monetary (구매금액) 지표**
- VIP 고객이 압도적으로 높은 구매금액을 보이며 가장 가치 있는 고객층으로, 충성 고객과 잠재 고객이 그 다음으로 높음

### 3. 매출 기여도 분석

- **잠재 고객 (39.9%)**: 전체 매출의 약 40%를 차지하는 핵심 성장 그룹으로, 고객 수가 많고 매출 기여도도 가장 높음
- **VIP 고객 (26.6%)**: 고객 비율에 비해 매우 높은 매출을 창출하는 최상위 고가치 고객층
- **충성 고객 (14.6%)**: 꾸준한 구매로 안정적인 매출을 담당하는 핵심 수익원
- **관심 필요 (9.1%)**: 과거에는 매출 기여도가 높았으나 최근 활동이 줄어든 그룹, 재활성화 필요
- **일반 고객 (5.1%)**: 중간 수준의 매출을 담당하는 그룹
- **이탈 위험 (2.7%)**: 매출 기여도가 매우 낮아 즉각적인 관리가 필요한 그룹
- **신규 고객 (2.1%)**: 최근 유입된 고객으로, 현재 매출 기여도는 낮으나 성장 가능성 있음

-> 전체 매출의 대부분이 잠재 고객, VIP 고객, 충성 고객에 집중되어 있으며, 하위 세그먼트의 활성화와 상위 그룹의 유지 관리가 중요합니다.

### 4. 세그먼트별 마케팅 전략 및 실행 방안

- **VIP 고객**: 프리미엄 VIP 프로그램 운영, 신제품 우선 체험 기회 제공, 개인 맞춤형 서비스와 전담 CS 지원
- **충성 고객**: 충성도 리워드 프로그램, 정기적인 혜택 제공, 교차 판매 및 업셀링 기회 확대
- **잠재 고객**: 구매 빈도와 금액 증대를 위한 타겟 프로모션, 개인화된 상품 추천, 단계적 혜택 제공
- **신규 고객**: 온보딩 프로그램 강화, 초기 구매 인센티브 제공, 제품 교육 및 사용법 가이드
- **관심 필요**: 맞춤형 재참여 캠페인, 특별 할인 혜택, 고객 피드백 수집 및 개선사항 반영
- **이탈 위험**: 긴급 윈백 캠페인, 대폭 할인 혜택, 이탈 사유 분석 및 개선책 마련
- **일반 고객**: 기본적인 마케팅 활동 유지, 점진적 세그먼트 상향 유도
